<a href="https://colab.research.google.com/github/PCBZ/CS6180-Course/blob/main/HW4_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install python-dotenv huggingface-hub llama-index transformers sentence-transformers llama-index-llms-huggingface llama-index-embeddings-huggingface pdfplumber llama-index-llms-openrouter llama-index-retrievers-bm25 tabula-py  jpype1 pystemmer
!apt-get install -y tesseract-ocr
!pip install pytesseract pymupdf
!apt-get install -y poppler-utils

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


## 1. Setup & Configuration
In this section, we set up the necessary dependencies, including third-party libraries and API keys.
We also configure the language models (LLMs) and embedding models used throughout the tutorial.

Key Steps:
- Import required Python libraries.
- Load API keys securely from environment variables.
- Initialize the OpenRouter LLMs for both querying and evaluation.
- Set up the HuggingFace embedding model for text representation.
- Apply `nest_asyncio` to handle event loop issues in Jupyter environments.


In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # Suppresses TensorFlow warnings
from dotenv import load_dotenv
import Stemmer
import nest_asyncio
import tabula
import pandas as pd
from dotenv import load_dotenv
from llama_index.core import Document
from llama_index.core import (SimpleDirectoryReader, VectorStoreIndex, Settings)
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.openrouter import OpenRouter
import asyncio


load_dotenv()  # Load environment variables from .env file
# ✅ Load API Key Securely (No Hardcoding!)
api_key = os.getenv("OPENROUTER_API_KEY")
if api_key:
    print("✅ API Key Loaded Successfully:", api_key[:5] + "..." + api_key[-5:])
else:
    print("⚠️ API Key is missing! Check your .env file.")


# ✅ Initialize OpenRouter LLM
llm = OpenRouter(api_key=api_key, model="meta-llama/llama-3.1-8b-instruct", max_tokens=512, context_window=4096)
Judge_llm = OpenRouter(api_key=api_key, model="qwen/qwen-turbo", max_tokens=512, context_window=4096)
Settings.llm = llm

# ✅ Apply nest_asyncio to fix event loop issues in Jupyter
nest_asyncio.apply()

# ✅ Set up embedding model
embed_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embed_model = HuggingFaceEmbedding(model_name=embed_model_name)
Settings.embed_model = embed_model


✅ API Key Loaded Successfully: sk-or...53eb0


## 2. Document Loading & Preprocessing
Here, we load the syllabus document from a PDF file and process it for retrieval.

Key Steps:
- Read the syllabus PDF and extract its textual content.
- Extract tabular data from the PDF using `tabula`.
- Convert the extracted table data into text format.
- Combine the extracted text and tables into a unified document.
- Define an ingestion pipeline to preprocess text by splitting it into manageable chunks and applying embeddings.


In [ ]:
import fitz
import pytesseract
from PIL import Image
import io

def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    ocr_text = []
    for page in doc:
        for image in page.get_images():
            try:
                image = Image.open(io.BytesIO(page.get_image_data(image[0])))
                text = pytesseract.image_to_string(image)
                ocr_text.append(text)
            except Exception as e:
                continue
    doc.close()
    return "\n".join(ocr_text)


# ✅ Load WebMD.pdf
pdf_path = "./WebMD.pdf"
documents = SimpleDirectoryReader(input_files=[pdf_path]).load_data()

# Convert table data into an additional Document
tables = tabula.read_pdf(pdf_path, pages="all")
table_docs = [df.to_markdown(index=False) for df in tables]
all_tables_text = "\n\n".join(table_docs)
document_from_tables = Document(text=all_tables_text)

# Combine original + table doc
documents = documents + [document_from_tables]

# Extract image using OCR
text_from_pdf = extract_text_from_pdf(pdf_path)
documents += [Document(text=text_from_pdf)]

# ✅ Create the pipeline with transformations
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.ingestion import IngestionPipeline

baseline_transformation = SentenceSplitter(chunk_size=256, chunk_overlap=0)
optimizer_transformation1 = SentenceSplitter(chunk_size=512, chunk_overlap=64)
optimizer_transformation2 = SentenceSplitter(chunk_size=512, chunk_overlap=64, paragraph_separator="\n\n")


pipeline = IngestionPipeline(transformations=[baseline_transformation, embed_model])

# # Run the pipeline
nodes = pipeline.run(documents=documents)


# =============================================================
# Task 5 - Improvement 1: Adaptive Chunking by Paragraph Length
# =============================================================

import re

def get_chunk_size_by_length(paragraph):
    word_count = len(paragraph.split())
    if word_count > 150:
        return 256
    elif word_count > 50:
        return 512
    else:
        return 1024

def adaptive_chunking_by_length(documents, embed_model):
    all_nodes = []

    for doc in documents:
        paragraphs = doc.text.split('\n\n')
        paragraphs = [p.strip() for p in paragraphs if len(p.strip()) > 20]

        for para in paragraphs:
            chunk_size = get_chunk_size_by_length(para)
            splitter = SentenceSplitter(chunk_size=chunk_size, chunk_overlap=min(chunk_size // 8, 64))
            para_doc = Document(text=para, metadata={**doc.metadata, "chunk_size_used": chunk_size})
            nodes = splitter.get_nodes_from_documents([para_doc])
            all_nodes.extend(nodes)

    for node in all_nodes:
        node.embedding = embed_model.get_text_embedding(node.text)

    return all_nodes

# nodes = adaptive_chunking_by_length(documents, embed_model)

# =========================================================
# Task 5 - Improvement 2: Conflict-Aware Document Ingestion
# =========================================================

def is_conflict_worthy(text):
    """
    Identify chunks worth checking for conflicts:
    - Contains numeric/statistical data
    - Contains medical claims
    """
    # Type 1: Numeric/statistical
    has_numeric = bool(re.search(r'\d+%|\d+ (days|hours|months|years|times)', text))

    # Type 2: Medical claims
    medical_claim_keywords = ["causes", "leads to", "results in", "is associated with", "increases risk", "linked to", "contributes to"]
    has_medical_claim = any(kw in text.lower() for kw in medical_claim_keywords)

    return has_numeric or has_medical_claim

def get_llm_knowledge(text, llm):
    """
    Ask LLM what it knows about the topic in the chunk
    """
    prompt = (
        f"In 2-3 sentences, what do you know about "
        f"the following medical claim? Be concise.\n\n"
        f"Claim: {text[:300]}"
    )
    response = llm.complete(prompt)
    return str(response)

def semantic_similarity(text1, text2, embed_model):
    """
    Compute cosine similarity between two texts
    """
    import numpy as np
    emb1 = np.array(embed_model.get_text_embedding(text1))
    emb2 = np.array(embed_model.get_text_embedding(text2))
    return float(np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2)))

def credibility_score(text, source=""):
    """
    Score credibility based on:
    - Source authority
    - Language certainty
    """
    score = 0.5
    text_lower = text.lower()

    # Source authority
    if "webmd" in source.lower():
        score += 0.3

    # Strong evidence language
    strong = {"study shows", "research confirms", "clinical trial", "evidence suggests", "according to", "published"}
    weak = {"some believe", "might", "unconfirmed", "rumored", "i think", "possibly"}

    for phrase in strong:
        if phrase in text_lower:
            score += 0.1
    for phrase in weak:
        if phrase in text_lower:
            score -= 0.1

    return min(max(score, 0.0), 1.0)

def handle_conflict(node, llm_knowledge, similarity_score):
    """
    Decide how to handle conflict based on:
    - Similarity between chunk and LLM knowledge
    - Credibility scores of both sides
    """
    source = node.metadata.get("source", "")
    chunk_score = credibility_score(node.text, source)
    llm_score = credibility_score(llm_knowledge)
    diff = chunk_score - llm_score

    if similarity_score > 0.85:
        # High similarity → no real conflict
        node.metadata["status"] = "consistent"
    elif diff > 0.2:
        # Chunk more credible → keep, mark as verified
        node.metadata["status"] = "verified"
    elif diff < -0.2:
        # LLM more credible → mark as disputed
        node.metadata["status"] = "disputed"
        node.metadata["disclaimer"] = (
            "⚠️ This information may conflict with "
            "established medical knowledge."
        )
    else:
        # Uncertain → keep both, mark as conflicting
        node.metadata["status"] = "conflicting"
        node.metadata["disclaimer"] = (
            "⚠️ Note: Conflicting information exists "
            "on this topic. Please verify with a "
            "medical professional."
        )

    return node

def apply_conflict_detection(nodes, llm, embed_model):
    """
    Main function: apply conflict detection to all
    conflict-worthy nodes
    """
    checked = 0
    for node in nodes:
        if is_conflict_worthy(node.text):
            # Get LLM's existing knowledge
            llm_knowledge = get_llm_knowledge(node.text, llm)
            # Compute semantic similarity
            sim = semantic_similarity(node.text, llm_knowledge, embed_model)
            # Handle conflict
            node = handle_conflict(node, llm_knowledge, sim)
            checked += 1

    print(f"✅ Conflict detection complete.")
    print(f"   Chunks checked: {checked}/{len(nodes)}")
    return nodes


# Apply to existing nodes
nodes = apply_conflict_detection(nodes, llm, embed_model)



✅ Conflict detection complete.
   Chunks checked: 10/73


## 3. Indexing & Retrieval
Once the document is processed, we create multiple retrievers for efficient information retrieval.

Key Steps:
- Build a `VectorStoreIndex` from the preprocessed document chunks.
- Implement different retrieval methods:
  - **Base Retriever:** Retrieves the most relevant document chunk.
  - **AutoMerging Retriever:** Aggregates multiple related chunks before returning results.
  - **BM25 Retriever:** Uses term frequency-based ranking (a traditional information retrieval method).
  - **Hybrid Fusion Retriever:** (Task 4)
- Define retrieval hyperparameters, such as similarity threshold and number of top results to return.


In [ ]:
# ✅ Create Vector Index and Query Engine
index = VectorStoreIndex(nodes)
query_engine = index.as_query_engine()

# ✅ Create base retrievers
base_retriever = index.as_retriever(similarity_top_k=1)

from llama_index.core.retrievers import AutoMergingRetriever
auto_base_retriever = index.as_retriever(similarity_top_k=3)
auto_merging_retriever = AutoMergingRetriever(auto_base_retriever, index.storage_context)

from llama_index.retrievers.bm25 import BM25Retriever


# Create Hybrid Fusion Retriever (Task 4)
from llama_index.core.retrievers import QueryFusionRetriever
bm25_retriever = BM25Retriever.from_defaults(nodes=nodes, similarity_top_k=2, stemmer=Stemmer.Stemmer("english"), language="english")
vector_retriever = index.as_retriever(similarity_top_k=5)

hybrid_fusion_retriever = QueryFusionRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    retriever_weights=[0.6, 0.4],
    similarity_top_k=3,
    num_queries=1,
    use_async=False
)


DEBUG:bm25s:Building index from IDs objects


Some nodes are missing content, skipping them...


## 4. Query Engines & Evaluation Setup
In this section, we configure the query engines and set up evaluation metrics to assess retriever performance.

Key Steps:
- Instantiate query engines for each retrieval method to allow direct querying.
- Define evaluation models to measure:
  - **Faithfulness:** Whether the retrieved information is accurate and grounded in the original document.
  - **Relevancy:** Whether the retrieved information is relevant to the query.
- Configure different retriever evaluators to assess retrieval effectiveness using metrics such as:
  - Mean Reciprocal Rank (MRR)
  - Hit Rate
  - Precision
  - Recall


In [ ]:
# ✅ Create query engines
from llama_index.core.query_engine import RetrieverQueryEngine

base_query_engine = RetrieverQueryEngine.from_args(base_retriever)
auto_query_engine = RetrieverQueryEngine.from_args(auto_merging_retriever)
bm25_query_engine = RetrieverQueryEngine.from_args(bm25_retriever)

hybrid_fusion_query_engine = RetrieverQueryEngine.from_args(hybrid_fusion_retriever)

# ✅ Initialize Evaluators
from llama_index.core.evaluation import FaithfulnessEvaluator, RelevancyEvaluator, RetrieverEvaluator

faithfulness_evaluator = FaithfulnessEvaluator(llm=Judge_llm)
relevancy_evaluator = RelevancyEvaluator(llm=Judge_llm)

# ✅ Define retriever evaluators
base_retriever_evaluator = RetrieverEvaluator.from_metric_names(["mrr", "hit_rate", "precision", "recall"], retriever=base_retriever)
auto_retriever_evaluator = RetrieverEvaluator.from_metric_names(["mrr", "hit_rate", "precision", "recall"], retriever=auto_merging_retriever)
bm25_retriever_evaluator = RetrieverEvaluator.from_metric_names(["mrr", "hit_rate", "precision", "recall"], retriever=bm25_retriever)
hybrid_fusion_retriever_evaluator = RetrieverEvaluator.from_metric_names(["mrr", "hit_rate", "precision", "recall"], retriever=hybrid_fusion_retriever)



## 5. Preparing Evaluation Questions & Display Functions

In this section, we define a set of evaluation questions to systematically assess the performance of different retrieval methods. Additionally, we create utility functions for displaying results in a structured and readable format.

Key Steps:
- Define a set of test queries (e.g., syllabus-related questions).
- Generate question-context pairs for automated evaluation.

To make evaluation results more interpretable, we define:
- **`displayify_df(df)`** – A helper function to format and display DataFrames neatly in Jupyter notebooks.
- **`display_retriever_eval_results(name, eval_results)`** – Computes and prints key retrieval evaluation metrics.

In [ ]:
# ✅ Use Evaluation Questions
# eval_questions = []
# with open('my_questions.txt', 'r') as file:
#     for line in file:
#         eval_questions.append(line.strip())

# ✅ Add more evaluation questions
# eval_questions = [
#     "What are the symptoms of chronic migraine?",
#     "How long does a migraine episode typically last?",
#     "What treatments are available for chronic migraine?"
# ]

eval_questions = [
    "How many days per month define chronic migraine?",
    "What percentage of chronic migraine patients have fibromyalgia?",
    "What is the age group most commonly affected by migraine?",
    "Is migraine associated with cardiovascular disease?",
    "What causes the transition from episodic to chronic migraine?",
    "Is botulinum toxin linked to migraine prevention?"
]


from llama_index.core.evaluation import generate_question_context_pairs
import os

if os.path.exists("qa_dataset.json"):
    os.remove("qa_dataset.json")
if os.path.exists("qa_dataset.json"):
    from llama_index.core.evaluation import EmbeddingQAFinetuneDataset
    qa_dataset = EmbeddingQAFinetuneDataset.from_json("qa_dataset.json")
    print("✅ Loaded qa_dataset from file")
else:
    qa_dataset = generate_question_context_pairs(nodes=nodes, llm=llm, num_questions_per_chunk=1)
    qa_dataset.save_json("qa_dataset.json")
    print("✅ Generated and saved qa_dataset")


# ✅ Pretty Display Function
def displayify_df(df):
    """For pretty displaying DataFrame in a notebook."""
    display_df = df.style.set_properties(
        **{
            "inline-size": "300px",
            "overflow-wrap": "break-word",
        }
    )
    display(display_df)

# Helper to display retrieval metrics
def display_retriever_eval_results(name, eval_results):
    """Build a small DataFrame summarizing retrieval metrics across queries."""
    print(f"=== {name} ===")
    metric_dicts = [res.metric_vals_dict for res in eval_results]
    if not metric_dicts:
        print("No retriever metrics found!")
        return
    df = pd.DataFrame(metric_dicts)
    #displayify_df(df)
    print("Mean:\n", df.mean(numeric_only=True), "\n")

100%|██████████| 73/73 [02:45<00:00,  2.26s/it]

✅ Generated and saved qa_dataset


## 6. Running Evaluations & Comparing Retrievers
Finally, we conduct retrieval evaluations and compare the effectiveness of different retrieval strategies.

Key Steps:
- Execute retrieval on all retrievers (Base, AutoMerging, BM25, Hybrid Fusion).
- Assess each retriever's faithfulness and relevancy scores.
- Compare retrieval results by averaging the evaluation metrics.
- Display results in tabular format for easy comparison.

This step helps determine which retrieval method performs best for a given dataset.


In [ ]:
# ✅ Modify Evaluation to Include Fusion Retriever
async def run_evaluation():
    eval_results = []

    for query in eval_questions:
        # Retrieve responses from all four retrievers
        base_response = base_query_engine.query(query)
        auto_response = auto_query_engine.query(query)
        bm25_response = bm25_query_engine.query(query)
        hybrid_response = hybrid_fusion_query_engine.query(query)

        base_text = base_response.response
        auto_text = auto_response.response
        bm25_text = bm25_response.response
        hybrid_text = hybrid_response.response

        base_contexts = "\n".join([node.get_content() for node in base_response.source_nodes])
        auto_contexts = "\n".join([node.get_content() for node in auto_response.source_nodes])
        bm25_contexts = "\n".join([node.get_content() for node in bm25_response.source_nodes])
        hybrid_contexts = "\n".join([node.get_content() for node in hybrid_response.source_nodes])


        # Evaluate Faithfulness & Relevancy for each retriever
        base_faithfulness = faithfulness_evaluator.evaluate_response(response=base_response)
        auto_faithfulness = faithfulness_evaluator.evaluate_response(response=auto_response)
        bm25_faithfulness = faithfulness_evaluator.evaluate_response(response=bm25_response)
        hybrid_faithfulness = faithfulness_evaluator.evaluate_response(response=hybrid_response)

        base_relevancy = relevancy_evaluator.evaluate_response(query=query, response=base_response)
        auto_relevancy = relevancy_evaluator.evaluate_response(query=query, response=auto_response)
        bm25_relevancy = relevancy_evaluator.evaluate_response(query=query, response=bm25_response)
        hybrid_relevancy = relevancy_evaluator.evaluate_response(query=query, response=hybrid_response)

        eval_results.append({
            "Query": query,
            "Base Response": base_text,
            "Auto-Merged Response": auto_text,
            "BM25 Response": bm25_text,
            "Hybrid Response": hybrid_text,
            "Base Context": "".join(base_contexts[:100]) + "... " + "".join(base_contexts[-100:]),
            "Auto Context": "".join(auto_contexts[:100]) + "... " + "".join(auto_contexts[-100:]),
            "BM25 Context": "".join(bm25_contexts[:100]) + "... " + "".join(bm25_contexts[-100:]),
            "Hybrid Context": "".join(hybrid_contexts[:100]) + "... " + "".join(hybrid_contexts[-100:]),
            "Base Faithfulness": base_faithfulness.score,
            "Auto Faithfulness": auto_faithfulness.score,
            "BM25 Faithfulness": bm25_faithfulness.score,
            "Hybrid Faithfulness": hybrid_faithfulness.score,
            "Base Relevancy": base_relevancy.score,
            "Auto Relevancy": auto_relevancy.score,
            "BM25 Relevancy": bm25_relevancy.score,
            "Hybrid Relevancy": hybrid_relevancy.score,
        })

    df = pd.DataFrame(eval_results)

    # Compute means for comparison
    print("\n✅ Faithfulness Comparison")
    print(f"Base Retriever: {df['Base Faithfulness'].mean():.4f}")
    print(f"Auto-Merging Retriever: {df['Auto Faithfulness'].mean():.4f}")
    print(f"BM25 Retriever: {df['BM25 Faithfulness'].mean():.4f}")
    print(f"Hybrid Retriever: {df['Hybrid Faithfulness'].mean():.4f}")

    print("\n✅ Relevancy Comparison")
    print(f"Base Retriever: {df['Base Relevancy'].mean():.4f}")
    print(f"Auto-Merging Retriever: {df['Auto Relevancy'].mean():.4f}")
    print(f"BM25 Retriever: {df['BM25 Relevancy'].mean():.4f}")
    print(f"Hybrid Retriever: {df['Hybrid Relevancy'].mean():.4f}")

    # Evaluate all retrievers on the QA dataset
    base_eval_results = await base_retriever_evaluator.aevaluate_dataset(qa_dataset)
    auto_eval_results = await auto_retriever_evaluator.aevaluate_dataset(qa_dataset)
    bm25_eval_results = await bm25_retriever_evaluator.aevaluate_dataset(qa_dataset)
    hybrid_eval_results = await hybrid_fusion_retriever_evaluator.aevaluate_dataset(qa_dataset)

    # Display

    print("=== Retrieval Metrics Comparison ===")
    display_retriever_eval_results("Base Retriever", base_eval_results)
    display_retriever_eval_results("Auto-Merging Retriever", auto_eval_results)
    display_retriever_eval_results("BM25 Retriever", bm25_eval_results)
    display_retriever_eval_results("Hybrid Retriever", hybrid_eval_results)

    # print("\n=== Per-Query Scores ===")
    # for _, row in df.iterrows():
    #     print(f"Q: {row['Query'][:60]}")
    #     print(f"   Base    F:{row['Base Faithfulness']:.2f} R:{row['Base Relevancy']:.2f}")
    #     print(f"   Auto    F:{row['Auto Faithfulness']:.2f} R:{row['Auto Relevancy']:.2f}")
    #     print(f"   BM25    F:{row['BM25 Faithfulness']:.2f} R:{row['BM25 Relevancy']:.2f}")
    #     print(f"   Hybrid  F:{row['Hybrid Faithfulness']:.2f} R:{row['Hybrid Relevancy']:.2f}")


    # Display results table
    # displayify_df(df)

# ✅ Execute Async Evaluation
asyncio.run(run_evaluation())


✅ Faithfulness Comparison
Base Retriever: 0.6667
Auto-Merging Retriever: 0.6667
BM25 Retriever: 1.0000
Hybrid Retriever: 1.0000

✅ Relevancy Comparison
Base Retriever: 0.6667
Auto-Merging Retriever: 0.6667
BM25 Retriever: 0.8333
Hybrid Retriever: 0.8333
=== Retrieval Metrics Comparison ===
=== Base Retriever ===
Mean:
 mrr          0.027397
hit_rate     0.027397
precision    0.027397
recall       0.027397
dtype: float64 

=== Auto-Merging Retriever ===
Mean:
 mrr          0.041096
hit_rate     0.054795
precision    0.018265
recall       0.054795
dtype: float64 

=== BM25 Retriever ===
Mean:
 mrr          0.020548
hit_rate     0.027397
precision    0.013699
recall       0.027397
dtype: float64 

=== Hybrid Retriever ===
Mean:
 mrr          0.029680
hit_rate     0.054795
precision    0.018265
recall       0.054795
dtype: float64 

